# Experimentos: cleaning visual vs dataset original

Tablero liviano para preparar la comparacion VSR. Este notebook no entrena modelos ni corre LLM.

In [1]:
import pandas as pd
from IPython.display import Markdown, display

from evaluation.src.visual_cleaning_notebook import (
    LLM_RESULT_FILES,
    RESULT_FILES,
    cargar_manifests,
    cargar_resultados_existentes,
    conclusion_automatica,
    distribucion_usabilidad,
    estado_archivos,
    perdida_por_fuente,
    resumen_resultados,
    tamanos_manifests,
)

pd.set_option("display.max_colwidth", 120)
manifests = cargar_manifests()

## 1. Manifests y tamanos

In [2]:
sizes = tamanos_manifests(manifests)
sizes

,manifest,clips
0,original_train,4826
1,original_val,466
2,original_test,658
3,visual_cleaned_train,4623
4,visual_cleaned_val,466
5,visual_cleaned_test_original,658


In [3]:
original_train = len(manifests["original_train"])
cleaned_train = len(manifests["visual_cleaned_train"])
pd.DataFrame([
    {"metric": "original_train", "clips": original_train},
    {"metric": "visual_cleaned_train", "clips": cleaned_train},
    {"metric": "excluded_train", "clips": original_train - cleaned_train},
])

,metric,clips
0,original_train,4826
1,visual_cleaned_train,4623
2,excluded_train,203


## 2. Distribucion visual

In [4]:
dist = distribucion_usabilidad(manifests)
dist.pivot_table(index="manifest", columns="training_usability", values="clips", fill_value=0, aggfunc="sum")

training_usability,bad_candidate,questionable,usable
manifest,,,
original_test,0,43,615
original_train,203,2260,2363
original_val,15,99,352
visual_cleaned_test_original,0,43,615
visual_cleaned_train,0,2260,2363
visual_cleaned_val,15,99,352


In [5]:
perdidas = perdida_por_fuente(manifests)
perdidas.head(20)

,source_id,original_train,visual_cleaned_train,excluded
0,ANÉCDOTA VIAJE MUNDIAL BRASIL 2014 parte 1,302,250,52
8,Las y los estudiantes al frente - Entrevista a la,257,207,50
17,PROFESIONES ARGENTINAS 🚖 Los SECRETOS de los TAXIS,73,57,16
14,PROFESIONES ARGENTINAS GOMEROS - Telefe Noticias,55,39,16
16,PROFESIONES ARGENTINAS KINESIÓLOGOS - Telefe Notic,62,50,12
12,PROFESIONES ARGENTINAS DIRECTORES de CASTING - Tel,87,77,10
23,Profesiones argentinas GASTROENTERÓLOGOS #TelefeNo,75,66,9
9,Los COCINEROS de TELEVISIÓN revelan sus SECRETOS e,99,91,8
15,PROFESIONES ARGENTINAS GUARDAESPALDAS - Telefe Not,83,75,8
22,Profesiones Argentinas PERIODISTAS DEPORTIVOS - Te,90,85,5


## 3. Resultados VSR

In [6]:
estado_archivos(RESULT_FILES)

,artifact,path,exists
0,baseline_original,C:\Users\bianc\NLP Natural Language Processing\labios-argentos\evaluation\outputs\visual_cleaning\results\baseline_o...,False
1,visual_cleaned,C:\Users\bianc\NLP Natural Language Processing\labios-argentos\evaluation\outputs\visual_cleaning\results\visual_cle...,False


In [7]:
resultados = cargar_resultados_existentes()
if resultados.empty:
    display(Markdown("**Pendiente de VM:** todavia no hay CSV VSR comparables en `evaluation/outputs/visual_cleaning/results/`."))
else:
    display(resultados.head())

**Pendiente de VM:** todavia no hay CSV VSR comparables en `evaluation/outputs/visual_cleaning/results/`.

In [8]:
if resultados.empty:
    display(Markdown("No se comparan WER/CER hasta traer resultados de VM."))
else:
    display(resumen_resultados(resultados))
    display(resumen_resultados(resultados, "training_usability"))
    display(resumen_resultados(resultados, "source_id"))

No se comparan WER/CER hasta traer resultados de VM.

## 4. Corrector LLM

In [9]:
llm_status = estado_archivos(LLM_RESULT_FILES)
display(llm_status)
if not llm_status["exists"].all():
    display(Markdown("**Bloqueado:** faltan outputs corregidos. El corrector LLM se evalua despues de tener predicciones VSR."))

,artifact,path,exists
0,baseline_original,C:\Users\bianc\NLP Natural Language Processing\labios-argentos\evaluation\outputs\visual_cleaning\llm_corrector\base...,False
1,visual_cleaned,C:\Users\bianc\NLP Natural Language Processing\labios-argentos\evaluation\outputs\visual_cleaning\llm_corrector\visu...,False


**Bloqueado:** faltan outputs corregidos. El corrector LLM se evalua despues de tener predicciones VSR.

In [10]:
display(Markdown(conclusion_automatica(manifests, resultados)))

Pendiente de VM: los manifests estan listos, pero todavia no hay resultados VSR comparables. La comparacion valida debe usar el test original completo.